# Clean Point-Level RCA Dataset Builder

This notebook rebuilds the clean point-level manufacturing dataset used by the RCA.

It performs:
- corrected merge logic with the run key `COIL + DATE`
- deterministic deduplication to one row per `COIL + DATE + MT`
- defect coverage auditing before label assignment
- leakage-safe point labels for `any_defect` plus the 50m, 200m, and 500m pre-defect windows
- validation output for row counts, duplicates, label balance, and dropped defect events


In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd


PRE_WINDOWS = (50, 200, 500)
RUN_KEY = ["COIL", "DATE"]
POINT_KEY = ["COIL", "DATE", "MT"]


def load_inputs(defects_path: Path, production_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load raw inputs and normalize the minimum run key to COIL + DATE."""
    defects = pd.read_csv(defects_path, low_memory=False)
    production = pd.read_csv(production_path, low_memory=False)

    for df in (defects, production):
        df["COIL"] = df["COIL"].astype("string").str.strip()
        df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce").dt.normalize()

    production["MT"] = pd.to_numeric(production["MT"], errors="coerce")
    defects["MT_FROM"] = pd.to_numeric(defects["MT_FROM"], errors="coerce")
    defects["MT_TO"] = pd.to_numeric(defects["MT_TO"], errors="coerce")

    # Keep a deterministic source order so tie-breaking is reproducible.
    production = production.assign(_source_order=np.arange(len(production), dtype=np.int64))

    defect_cols = [c for c in defects.columns if c.startswith("DIF_TIPO_")]
    for col in defect_cols:
        defects[col] = pd.to_numeric(defects[col], errors="coerce").fillna(0).astype("Int64")

    # Drop unusable rows before any merge logic.
    production = production.dropna(subset=POINT_KEY).copy()
    defects = defects.dropna(subset=RUN_KEY + ["MT_FROM", "MT_TO"]).copy()

    # Normalise interval direction if any rows arrive reversed.
    swap_mask = defects["MT_FROM"] > defects["MT_TO"]
    if swap_mask.any():
        defects.loc[swap_mask, ["MT_FROM", "MT_TO"]] = defects.loc[
            swap_mask, ["MT_TO", "MT_FROM"]
        ].to_numpy()

    return defects, production


def deduplicate_production(production: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse duplicate physical positions to one row per COIL + DATE + MT.

    Deterministic rule:
    - prefer the latest TIME_START_PROCESS within the key
    - break ties by original row order
    """
    prod = production.copy()
    prod["_time_order"] = pd.to_timedelta(prod.get("TIME_START_PROCESS"), errors="coerce")

    prod = prod.sort_values(
        POINT_KEY + ["_time_order", "_source_order"],
        ascending=[True, True, True, True, True],
        na_position="first",
        kind="stable",
    )
    prod = prod.drop_duplicates(subset=POINT_KEY, keep="last").copy()
    prod = prod.sort_values(POINT_KEY, kind="stable").reset_index(drop=True)
    prod = prod.drop(columns=["_time_order", "_source_order"])
    return prod


def audit_defect_coverage(
    defects: pd.DataFrame, production_clean: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Flag defect events by observed PLC coverage after deduplication.

    Coverage is checked against the cleaned point-level table so labels are only
    applied where PLC rows actually exist.
    """
    run_mt_lookup: dict[tuple[str, pd.Timestamp], np.ndarray] = {}
    for key, group in production_clean.groupby(RUN_KEY, sort=False):
        run_mt_lookup[key] = group["MT"].to_numpy(dtype=float, copy=True)

    audited = defects.copy().reset_index(drop=True)
    audited["defect_event_id"] = np.arange(len(audited), dtype=np.int64)
    audited["coverage_status"] = "missing_run"
    audited["points_in_interval"] = 0
    audited["run_min_mt"] = np.nan
    audited["run_max_mt"] = np.nan

    for idx, defect in audited.iterrows():
        key = (defect["COIL"], defect["DATE"])
        mt = run_mt_lookup.get(key)
        if mt is None or len(mt) == 0:
            continue

        mt_from = float(defect["MT_FROM"])
        mt_to = float(defect["MT_TO"])

        left = np.searchsorted(mt, mt_from, side="left")
        right = np.searchsorted(mt, mt_to, side="right")
        points_in_interval = int(right - left)

        run_min_mt = float(mt[0])
        run_max_mt = float(mt[-1])

        if points_in_interval == 0:
            status = "zero_coverage"
        elif mt_from < run_min_mt or mt_to > run_max_mt:
            status = "partial_coverage"
        else:
            status = "full_coverage"

        audited.at[idx, "coverage_status"] = status
        audited.at[idx, "points_in_interval"] = points_in_interval
        audited.at[idx, "run_min_mt"] = run_min_mt
        audited.at[idx, "run_max_mt"] = run_max_mt

    valid_defects = audited.loc[audited["points_in_interval"] > 0].copy()
    return audited, valid_defects


def assign_point_labels(
    production_clean: pd.DataFrame,
    valid_defects: pd.DataFrame,
    pre_windows: tuple[int, ...] = PRE_WINDOWS,
) -> pd.DataFrame:
    """
    Rebuild point-level labels after key correction, deduplication, and coverage checks.

    Features are the current-row PLC readings only. No segment-level summaries are added.
    """
    df_clean = production_clean.copy()
    df_clean["any_defect"] = 0
    for window in pre_windows:
        df_clean[f"pre_any_defect_{window}m"] = 0

    if valid_defects.empty:
        return df_clean

    valid_defects = valid_defects.sort_values(RUN_KEY + ["MT_FROM", "MT_TO"], kind="stable")
    grouped_defects = valid_defects.groupby(RUN_KEY, sort=False)

    for key, row_index in df_clean.groupby(RUN_KEY, sort=False).groups.items():
        if key not in grouped_defects.groups:
            continue

        run_rows = np.asarray(list(row_index), dtype=np.int64)
        mt = df_clean.loc[run_rows, "MT"].to_numpy(dtype=float)
        run_min_mt = float(mt.min())

        any_defect = np.zeros(len(run_rows), dtype=np.uint8)
        pre_labels = {window: np.zeros(len(run_rows), dtype=np.uint8) for window in pre_windows}

        defects_for_run = valid_defects.loc[grouped_defects.groups[key]]
        for defect in defects_for_run.itertuples(index=False):
            mt_from = float(defect.MT_FROM)
            mt_to = float(defect.MT_TO)

            # Defect rows are inclusive on both interval ends.
            in_defect = (mt >= mt_from) & (mt <= mt_to)
            any_defect[in_defect] = 1

            # Pre-window labels are strictly prior to the defect start.
            for window in pre_windows:
                pre_start = max(run_min_mt, mt_from - window)
                pre_mask = (mt >= pre_start) & (mt < mt_from)
                pre_labels[window][pre_mask] = 1

        df_clean.loc[run_rows, "any_defect"] = any_defect
        for window in pre_windows:
            df_clean.loc[run_rows, f"pre_any_defect_{window}m"] = pre_labels[window]

    return df_clean


def validate_output(df_clean: pd.DataFrame, defect_audit: pd.DataFrame) -> dict[str, object]:
    unique_points = int(df_clean.drop_duplicates(subset=POINT_KEY).shape[0])
    duplicate_count = int(len(df_clean) - unique_points)

    validation = {
        "row_count": int(len(df_clean)),
        "unique_point_count": unique_points,
        "duplicate_point_count": duplicate_count,
        "label_distribution": {
            "any_defect": {
                "positive_rows": int(df_clean["any_defect"].sum()),
                "positive_rate": float(df_clean["any_defect"].mean()),
            },
            "pre_any_defect_50m": {
                "positive_rows": int(df_clean["pre_any_defect_50m"].sum()),
                "positive_rate": float(df_clean["pre_any_defect_50m"].mean()),
            },
            "pre_any_defect_200m": {
                "positive_rows": int(df_clean["pre_any_defect_200m"].sum()),
                "positive_rate": float(df_clean["pre_any_defect_200m"].mean()),
            },
            "pre_any_defect_500m": {
                "positive_rows": int(df_clean["pre_any_defect_500m"].sum()),
                "positive_rate": float(df_clean["pre_any_defect_500m"].mean()),
            },
        },
        "defect_events_dropped_due_to_zero_coverage": int(
            (defect_audit["points_in_interval"] == 0).sum()
        ),
        "defect_coverage_breakdown": defect_audit["coverage_status"].value_counts().to_dict(),
    }
    return validation


def print_validation(validation: dict[str, object]) -> None:
    print(f"Row count: {validation['row_count']}")
    print(f"Unique (COIL, DATE, MT): {validation['unique_point_count']}")
    print(f"Duplicate count: {validation['duplicate_point_count']}")

    label_distribution = validation["label_distribution"]
    print("Label distribution:")
    for label_name in [
        "any_defect",
        "pre_any_defect_50m",
        "pre_any_defect_200m",
        "pre_any_defect_500m",
    ]:
        stats = label_distribution[label_name]
        print(
            f"  {label_name}: {stats['positive_rows']} positive rows "
            f"({stats['positive_rate']:.6%})"
        )

    print(
        "Number of defect events dropped due to zero coverage: "
        f"{validation['defect_events_dropped_due_to_zero_coverage']}"
    )
    print("Defect coverage breakdown:")
    for status, count in validation["defect_coverage_breakdown"].items():
        print(f"  {status}: {count}")


def build_clean_point_dataset(
    defects_path: Path,
    production_path: Path,
    output_path: Path | None = None,
    audit_path: Path | None = None,
    validation_path: Path | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, object]]:
    defects_raw, production_raw = load_inputs(defects_path, production_path)
    production_clean = deduplicate_production(production_raw)
    defect_audit, valid_defects = audit_defect_coverage(defects_raw, production_clean)
    df_clean = assign_point_labels(production_clean, valid_defects)
    validation = validate_output(df_clean, defect_audit)

    if output_path is not None:
        df_clean.to_csv(output_path, index=False)
    if audit_path is not None:
        defect_audit.to_csv(audit_path, index=False)
    if validation_path is not None:
        validation_path.write_text(json.dumps(validation, indent=2))

    return df_clean, defect_audit, validation


def parse_args() -> argparse.Namespace:
    default_root = Path.cwd()
    parser = argparse.ArgumentParser(
        description="Build a clean point-level defect dataset with corrected merge logic."
    )
    parser.add_argument(
        "--production",
        type=Path,
        default=default_root / "RC_PRODUCTION_clean.csv",
        help="Path to the raw PLC production CSV.",
    )
    parser.add_argument(
        "--defects",
        type=Path,
        default=default_root / "RC_DEFECTS_clean.csv",
        help="Path to the raw defects CSV.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=default_root / "rca_point_level_clean.csv",
        help="Where to save the cleaned point-level CSV.",
    )
    parser.add_argument(
        "--audit-output",
        type=Path,
        default=default_root / "rca_defect_coverage_audit.csv",
        help="Where to save the defect coverage audit CSV.",
    )
    parser.add_argument(
        "--validation-output",
        type=Path,
        default=default_root / "rca_build_validation.json",
        help="Where to save the validation summary JSON.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    df_clean, defect_audit, validation = build_clean_point_dataset(
        defects_path=args.defects,
        production_path=args.production,
        output_path=args.output,
        audit_path=args.audit_output,
        validation_path=args.validation_output,
    )

    print_validation(validation)
    print(f"Clean dataset saved to: {args.output}")
    print(f"Defect audit saved to: {args.audit_output}")
    print(f"Validation summary saved to: {args.validation_output}")
    print(f"df_clean shape: {df_clean.shape}")
    print(f"defect_audit shape: {defect_audit.shape}")



## Run The Rebuild

Execute the next cell to rebuild the clean point-level dataset.

If your raw defect and production CSV files are stored in a different folder, update `defects_path` and `production_path` in the next cell before running it.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

def first_existing(*candidates: str) -> Path:
    for candidate in candidates:
        path = PROJECT_DIR / candidate
        if path.exists():
            return path
    raise FileNotFoundError(
        "Update `defects_path` and `production_path` in this cell to match the location of your raw CSV files."
    )


defects_path = first_existing("RC_DEFECTS_clean.csv")
production_path = first_existing("RC_PRODUCTION_clean.csv", "RC_PRODUCTION_clean(1).csv")
output_path = PROJECT_DIR / "rca_point_level_clean.csv"
audit_path = PROJECT_DIR / "rca_defect_coverage_audit.csv"
validation_path = PROJECT_DIR / "rca_build_validation.json"

df_clean, defect_audit, validation = build_clean_point_dataset(
    defects_path=defects_path,
    production_path=production_path,
    output_path=output_path,
    audit_path=audit_path,
    validation_path=validation_path,
)

print_validation(validation)
df_clean.head()
